# Teste do TEU detetor de bola + tempo util
Usa o `best.pt` que treinaste (na Drive) para detetar a bola no teu video Court-Master, e corta o tempo util pela bola.
`Runtime -> T4 GPU`.

## 1. Setup

In [ ]:
!pip install -q ultralytics
from google.colab import drive; drive.mount('/content/drive')
from ultralytics import YOLO
MODEL='/content/drive/MyDrive/PadelPro_Vision/weights/ball_yolo/best.pt'
VIDEO='/content/drive/MyDrive/PadelPro_Vision/videos/Analise Padel Modelo - Parada 4 min.mp4'
import os; print('modelo?',os.path.exists(MODEL),'| video?',os.path.exists(VIDEO))
model=YOLO(MODEL)

## 2. TESTE visual (domain gap 1080p->540p)
Mostra a bola detetada em 24 frames. Se acertar na maioria -> seguimos. Se nao -> a camara/resolucao e' o limite.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
cap=cv2.VideoCapture(VIDEO); n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
idxs=[int(n*f) for f in np.linspace(0.18,0.95,24)]; imgs=[]; hits=0
for i in idxs:
    cap.set(cv2.CAP_PROP_POS_FRAMES,i); ok,fr=cap.read()
    if not ok: continue
    r=model.predict(fr, conf=0.25, imgsz=960, verbose=False)[0]
    if len(r.boxes): hits+=1
    for b in r.boxes:
        x1,y1,x2,y2=map(int,b.xyxy[0]); c=float(b.conf[0])
        cv2.rectangle(fr,(x1-6,y1-6),(x2+6,y2+6),(0,0,255),2); cv2.putText(fr,f'{c:.2f}',(x1,y1-8),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,0,255),2)
    imgs.append(cv2.cvtColor(fr,cv2.COLOR_BGR2RGB))
cap.release(); print(f'bola detetada em {hits}/{len(imgs)} frames')
fig,ax=plt.subplots(4,6,figsize=(20,11))
for a,im in zip(ax.ravel(),imgs): a.imshow(im); a.axis('off')
plt.tight_layout(); plt.show()

## 3. (se o teste for bom) Bola por frame em todo o video

In [ ]:
cap=cv2.VideoCapture(VIDEO); fps=cap.get(cv2.CAP_PROP_FPS); n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
bola=[False]*n; i=0
while True:
    ok,fr=cap.read()
    if not ok: break
    r=model.predict(fr, conf=0.25, imgsz=960, verbose=False)[0]
    bola[i]=len(r.boxes)>0; i+=1
cap.release()
print(f'bola visivel em {sum(bola)}/{n} ({100*sum(bola)/n:.0f}%)')

## 4. Rallies (regras v9) + cortar tempo util

In [ ]:
def segmentar(bola,fps,gap_fora_s=2.0,pre=1.5,pos=2.5,minr=1.0):
    n=len(bola); g=int(gap_fora_s*fps); pr=int(pre*fps); po=int(pos*fps); ml=int(minr*fps)
    runs=[]; i=0
    while i<n:
        if not bola[i]: i+=1; continue
        a=i
        while i+1<n and bola[i+1]: i+=1
        runs.append([a,i]); i+=1
    bl=[]
    for r in runs:
        if bl and (r[0]-bl[-1][1]-1)<=g: bl[-1][1]=r[1]
        else: bl.append(r)
    segs=[[max(0,a-pr),min(n-1,b+po)] for a,b in bl if b-a+1>=ml]
    for k in range(len(segs)-1):
        if segs[k][1]>=segs[k+1][0]:
            m=(segs[k][1]+segs[k+1][0])//2; segs[k][1]=m; segs[k+1][0]=m+1
    return segs
segs=segmentar(bola,fps); dur=[(b-a)/fps for a,b in segs]
print(f'{len(segs)} rallies | tempo util {sum(dur):.0f}s/{n/fps:.0f}s ({100*sum(dur)/(n/fps):.0f}%) | media {np.mean(dur):.1f}s' if segs else 'sem rallies')
import subprocess,os
os.makedirs('/content/clips',exist_ok=True); parts=[]
for j,(a,b) in enumerate(segs):
    o=f'/content/clips/r{j:02d}.mp4'
    subprocess.run(['ffmpeg','-y','-ss',f'{a/fps:.2f}','-to',f'{b/fps:.2f}','-i',VIDEO,'-c:v','libx264','-crf','23','-preset','veryfast','-an',o],capture_output=True)
    parts.append(os.path.abspath(o))
open('/content/list.txt','w').write('\n'.join(f"file '{p}'" for p in parts))
OUT='/content/drive/MyDrive/PadelPro_Vision/jogos/parada4_teste/TempoUtil_bola.mp4'
os.makedirs(os.path.dirname(OUT),exist_ok=True)
subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i','/content/list.txt','-c','copy',OUT],capture_output=True)
print('condensado na Drive:', os.path.exists(OUT))